# 9. Point Track Visualizer

Overlay the raw RGB frames with the tracked points stored in `meta_data/meata_data.npz`, similar to the TAPIR or CoTracker demos. Configure the paths below, then execute each cell in order to produce the visualization grid.


In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#A0A0A0"
plt.rcParams["axes.labelcolor"] = "#333333"
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "semibold"
plt.rcParams["font.size"] = 11


In [ ]:
# --- User configuration ----------------------------------------------------

META_DATA_PATH = Path("/home/justin/code/point-to-pose/debug/pipeline/meta_data/meata_data.npz")

# Option A: HO3D dataset sequence (preferred)
USE_HO3D_SEQUENCE = True
HO3D_SEQUENCE_DIR = Path("/home/justin/data/HO3D_V3/evaluation/AP10")
HO3D_RGB_SUBDIR = "rgb"
HO3D_RGB_EXTENSIONS = (".jpg", ".png")

# Option B: Manual template or folder (used when USE_HO3D_SEQUENCE=False)
IMAGE_PATH_TEMPLATE = None  # e.g., "/path/to/rgb/frame_{frame_id:06d}.png" or "/path/to/rgb"

START_FRAME = 240           # Frame id or (if missing) the row index to start at
NUM_FRAMES = 15             # How many consecutive frames to render
TREAT_START_AS_FRAME_ID = True
FILTER_POINTS_BY_START_VISIBILITY = True

POINT_SIZE = 60            # Scatter marker size
POINT_EDGE_COLOR = "#f8f8f8"
POINT_ALPHA = 0.95
MAX_COLUMNS = 4            # Grid width when plotting multiple frames

TRACE_VIDEO_OUTPUT = Path("/home/justin/code/point-to-pose/debug/track_viz/trace_evolution.mp4")
TRACE_VIDEO_FPS = 5
TRACE_VIDEO_LINE_WIDTH = 3
TRACE_VIDEO_POINT_SIZE = 8

TRACE_UNCERT_VIDEO_OUTPUT = Path("/home/justin/code/point-to-pose/debug/track_viz/trace_uncertainty.mp4")
TRACE_UNCERT_CMAP = "inferno"
TRACE_UNCERT_COLOR_LIMITS = None  # Set to (min, max) to override auto-scaling
TRACE_UNCERT_LEGEND_WIDTH = 50
TRACE_UNCERT_LEGEND_MARGIN = 10
TRACE_UNCERT_LEGEND_FONT_SCALE = 0.5

print(f"Meta-data file: {META_DATA_PATH}")
print(f"HO3D sequence:   {HO3D_SEQUENCE_DIR if USE_HO3D_SEQUENCE else 'disabled'}")
print(f"Image template:  {IMAGE_PATH_TEMPLATE}")
print(f"Trace video out: {TRACE_VIDEO_OUTPUT}")
print(f"Trace uncertainty video out: {TRACE_UNCERT_VIDEO_OUTPUT}")


In [ ]:
HO3D_RGB_FILES: List[Path] = []
HO3D_FRAME_ID_TO_INDEX: Dict[int, int] = {}


def _reconstruct_ragged(np_data, field_name: str, element_dims: Optional[Sequence[int]] = None):
    data_key = f"{field_name}_data"
    if data_key not in np_data:
        return None

    data_flat = np_data[data_key]
    offsets = np_data[f"{field_name}_offsets"]
    lengths = np_data[f"{field_name}_lengths"]
    element_size = None
    if element_dims is not None:
        element_size = int(np.prod(element_dims))

    sequences = []
    for offset, length in zip(offsets, lengths):
        segment = data_flat[offset : offset + length]
        if element_size is not None:
            if length % element_size != 0:
                raise ValueError(
                    f"Field '{field_name}' length {length} is not divisible by element size {element_size}."
                )
            if length == 0:
                segment = np.empty((0, *element_dims), dtype=segment.dtype)
            else:
                segment = segment.reshape(-1, *element_dims)
        else:
            segment = segment.copy()
        sequences.append(segment)

    return sequences


def _extract_frame_ids(np_data, fallback_length: int) -> np.ndarray:
    for key in ("frame_id", "frame_ids"):
        if key in np_data:
            arr = np.asarray(np_data[key]).reshape(-1)
            return arr.astype(int)
    return np.arange(fallback_length, dtype=int)


def load_meta_data(meta_data_path: Path) -> Dict[str, List[np.ndarray]]:
    np_data = np.load(meta_data_path, allow_pickle=True)

    track2d = _reconstruct_ragged(np_data, "track2d", element_dims=(2,))
    visibles = _reconstruct_ragged(np_data, "visibles")
    uncertainties = _reconstruct_ragged(np_data, "uncertainties")

    frame_count = len(track2d) if track2d is not None else len(visibles)
    frame_ids = _extract_frame_ids(np_data, frame_count)

    def _ensure_bool_list(seq, length):
        if seq is None:
            return [np.zeros((0,), dtype=bool) for _ in range(length)]
        return [np.asarray(v).astype(bool) for v in seq]

    def _ensure_float_list(seq, length):
        if seq is None:
            return [np.zeros((0,), dtype=float) for _ in range(length)]
        return [np.asarray(v).astype(float) for v in seq]

    meta = {
        "frame_ids": frame_ids,
        "track2d": track2d or [np.zeros((0, 2), dtype=float) for _ in range(frame_count)],
        "visibles": _ensure_bool_list(visibles, frame_count),
        "uncertainties": _ensure_float_list(uncertainties, frame_count),
    }

    return meta


def discover_ho3d_sequence(
    sequence_dir: Path,
    rgb_subdir: str = "rgb",
    extensions: Sequence[str] = (".jpg", ".png"),
) -> tuple[list[Path], Dict[int, int]]:
    rgb_dir = sequence_dir / rgb_subdir
    if not rgb_dir.exists():
        raise FileNotFoundError(f"RGB directory not found: {rgb_dir}")

    files: list[Path] = []
    extensions = tuple(ext.lower() for ext in extensions)
    for ext in extensions:
        files.extend(sorted(rgb_dir.glob(f"*{ext}")))
    files = sorted(set(files))

    if len(files) == 0:
        raise FileNotFoundError(
            f"No RGB files with extensions {extensions} found under {rgb_dir}"
        )

    frame_lookup: Dict[int, int] = {}
    for idx, file in enumerate(files):
        stem = file.stem
        digits = "".join(ch for ch in stem if ch.isdigit())
        if digits:
            try:
                frame_lookup.setdefault(int(digits), idx)
            except ValueError:
                continue

    return files, frame_lookup


def resolve_ho3d_frame_path(frame_id: int) -> Path:
    if not HO3D_RGB_FILES:
        raise FileNotFoundError("HO3D RGB files not initialized. Run the setup cell.")

    if frame_id in HO3D_FRAME_ID_TO_INDEX:
        idx = HO3D_FRAME_ID_TO_INDEX[frame_id]
    else:
        idx = max(0, min(frame_id, len(HO3D_RGB_FILES) - 1))
    return HO3D_RGB_FILES[idx]


def resolve_image_path(image_path: str | Path, frame_id: int) -> Path:
    """Resolve the actual file path for a given frame id (non-HO3D fallback)."""

    if image_path is None:
        raise FileNotFoundError(
            "IMAGE_PATH_TEMPLATE is None. Set it in the config cell or enable HO3D loading."
        )

    image_path = str(image_path)
    if "{frame_id" in image_path:
        return Path(image_path.format(frame_id=frame_id))

    path = Path(image_path)
    if path.is_dir():
        default_templates = [f"{frame_id:06d}", f"{frame_id:05d}", str(frame_id)]
        extensions = (".png", ".jpg", ".jpeg", ".bmp")
        for stem in default_templates:
            for ext in extensions:
                candidate = path / f"{stem}{ext}"
                if candidate.exists():
                    return candidate
        # Fallback: index into sorted images if direct lookup fails
        sorted_frames = sorted(
            [p for p in path.iterdir() if p.suffix.lower() in extensions]
        )
        if len(sorted_frames) == 0:
            raise FileNotFoundError(f"No images with extensions {extensions} under {path}")
        idx = min(max(frame_id, 0), len(sorted_frames) - 1)
        return sorted_frames[idx]

    if path.exists():
        return path

    raise FileNotFoundError(
        f"Could not resolve image path for frame {frame_id}: '{image_path}'"
    )


def load_rgb_frame(image_path: str | Path | None, frame_id: int) -> np.ndarray:
    if USE_HO3D_SEQUENCE:
        frame_file = resolve_ho3d_frame_path(frame_id)
    else:
        frame_file = resolve_image_path(image_path, frame_id)

    bgr = cv2.imread(str(frame_file), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Unable to load image at {frame_file}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb



In [ ]:
if USE_HO3D_SEQUENCE:
    HO3D_RGB_FILES, HO3D_FRAME_ID_TO_INDEX = discover_ho3d_sequence(
        HO3D_SEQUENCE_DIR,
        rgb_subdir=HO3D_RGB_SUBDIR,
        extensions=HO3D_RGB_EXTENSIONS,
    )
    preview = HO3D_RGB_FILES[:2]
    print(f"Loaded {len(HO3D_RGB_FILES)} HO3D RGB frames from {HO3D_SEQUENCE_DIR}")
    if preview:
        print("Sample files:")
        for path in preview:
            print(f"  - {path}")
else:
    HO3D_RGB_FILES = []
    HO3D_FRAME_ID_TO_INDEX = {}
    print("HO3D loading disabled; using IMAGE_PATH_TEMPLATE fallback.")


In [ ]:
def get_start_index(frame_ids: np.ndarray, start_frame: int, treat_as_id: bool) -> int:
    if treat_as_id:
        matches = np.where(frame_ids == start_frame)[0]
        if len(matches) > 0:
            return int(matches[0])
    # Fallback: clamp to valid index range
    start_idx = int(start_frame)
    return max(0, min(start_idx, len(frame_ids) - 1))


def choose_track_ids(meta: Dict[str, List[np.ndarray]], start_idx: int, by_visibility: bool) -> np.ndarray:
    tracks = meta["track2d"][start_idx]
    if tracks.size == 0:
        return np.array([], dtype=int)

    if by_visibility and meta["visibles"]:
        visible_mask = meta["visibles"][start_idx]
        track_ids = np.where(visible_mask)[0]
    else:
        track_ids = np.arange(tracks.shape[0])
    return track_ids.astype(int)


def make_color_map(track_ids: Sequence[int], cmap_name: str = "hsv") -> Dict[int, np.ndarray]:
    if len(track_ids) == 0:
        return {}
    cmap = plt.get_cmap(cmap_name, len(track_ids))
    colors = {}
    for idx, track_id in enumerate(track_ids):
        colors[track_id] = np.array(cmap(idx)[:3])  # RGB in 0-1 range
    return colors


def _prep_axes(num_items: int, max_cols: int) -> np.ndarray:
    ncols = min(max(num_items, 1), max_cols) if num_items > 0 else 1
    nrows = math.ceil(num_items / ncols) if num_items > 0 else 1
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 4.2 * nrows))
    if isinstance(axes, np.ndarray):
        axes_flat = axes.ravel()
    else:
        axes_flat = np.array([axes])
    # Hide any unused axes later
    return axes_flat


def visualize_sequence(
    meta: Dict[str, List[np.ndarray]],
    image_path: str | Path | None,
    start_idx: int,
    num_frames: int,
    track_ids: Sequence[int],
    point_size: float = 50,
    alpha: float = 0.9,
    edge_color: str = "white",
    max_cols: int = 4,
):
    track_ids = np.asarray(track_ids, dtype=int)
    color_map = make_color_map(track_ids)

    total_frames = len(meta["track2d"])
    frame_range = range(start_idx, min(start_idx + num_frames, total_frames))
    axes = _prep_axes(len(frame_range), max_cols)

    for ax in axes[len(frame_range) :]:
        ax.axis("off")

    for ax, frame_idx in zip(axes, frame_range):
        frame_id = int(meta["frame_ids"][frame_idx]) if meta["frame_ids"].size else frame_idx
        try:
            image = load_rgb_frame(image_path, frame_id)
        except FileNotFoundError as exc:
            ax.set_title(f"Frame {frame_id} (image missing)")
            ax.text(0.5, 0.5, str(exc), ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            continue

        points = meta["track2d"][frame_idx]
        visibles = meta["visibles"][frame_idx]
        if points.size == 0:
            ax.imshow(image)
            ax.set_title(f"Frame {frame_id}: no tracks")
            ax.axis("off")
            continue

        if len(visibles) == 0:
            visible_mask = np.ones((points.shape[0],), dtype=bool)
        else:
            visible_mask = visibles.astype(bool)

        selected_ids = track_ids
        if selected_ids.size == 0:
            selected_ids = np.where(visible_mask)[0]

        selected_ids = [tid for tid in selected_ids if tid < points.shape[0] and visible_mask[tid]]
        if not selected_ids:
            ax.imshow(image)
            ax.set_title(f"Frame {frame_id}: no selected tracks visible")
            ax.axis("off")
            continue

        track_points = points[selected_ids]
        colors = np.array([color_map.get(tid, (1.0, 0.2, 0.2)) for tid in selected_ids])

        ax.imshow(image)
        ax.scatter(
            track_points[:, 0],
            track_points[:, 1],
            s=point_size,
            c=colors,
            alpha=alpha,
            edgecolors=edge_color,
            linewidths=0.6,
        )
        ax.set_title(f"Frame {frame_id}")
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()


def plot_traces_on_last_frame(
    meta: Dict[str, List[np.ndarray]],
    image_path: str | Path | None,
    start_idx: int,
    num_frames: int,
    track_ids: Sequence[int],
    color_map: Optional[Dict[int, np.ndarray]] = None,
    linewidth: float = 2.0,
    point_size: float = 80,
    alpha: float = 0.95,
):
    total_frames = len(meta["track2d"])
    frame_range = range(start_idx, min(start_idx + num_frames, total_frames))
    if len(frame_range) == 0:
        raise ValueError("No frames available for the requested range.")

    last_idx = frame_range[-1]
    frame_id = int(meta["frame_ids"][last_idx]) if meta["frame_ids"].size else last_idx
    image = load_rgb_frame(image_path, frame_id)

    if color_map is None:
        color_map = make_color_map(track_ids)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image)

    track_ids = np.asarray(track_ids, dtype=int)
    if track_ids.size == 0:
        track_ids = np.arange(meta["track2d"][start_idx].shape[0])

    for tid in track_ids:
        traj = []
        for idx in frame_range:
            points = meta["track2d"][idx]
            if tid >= points.shape[0]:
                continue

            visible = meta["visibles"][idx]
            if len(visible) == 0:
                is_visible = True
            else:
                vis_arr = visible.astype(bool)
                if tid >= len(vis_arr) or not vis_arr[tid]:
                    continue
                is_visible = True

            if is_visible:
                traj.append(points[tid])
        if len(traj) < 2:
            continue
        traj = np.stack(traj)
        color = color_map.get(tid, np.array([1.0, 0.2, 0.2]))
        ax.plot(traj[:, 0], traj[:, 1], color=color, linewidth=linewidth, alpha=alpha)
        ax.scatter(
            traj[-1, 0],
            traj[-1, 1],
            s=point_size,
            color=color,
            edgecolors="white",
            linewidths=0.6,
        )

    ax.set_title(f"Traces through frame {frame_id}")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()



In [ ]:
def _create_uncertainty_legend(
    height: int,
    legend_width: int,
    cmap,
    limits: Tuple[float, float],
    font_scale: float = 0.5,
) -> np.ndarray:
    legend = np.zeros((height, legend_width, 3), dtype=np.uint8)
    u_min, u_max = limits
    for y in range(height):
        t = 1.0 - y / max(height - 1, 1)
        value = u_min + t * (u_max - u_min)
        color = cmap((value - u_min) / (u_max - u_min + 1e-8))[:3]
        bgr = tuple(int(c * 255) for c in color[::-1])
        legend[y, :] = bgr

    # Add labels (top = max, bottom = min)
    label_color = (255, 255, 255)
    thickness = max(1, int(font_scale))
    cv2.putText(
        legend,
        f"{u_max:.3f}",
        (2, int(12 * font_scale) + 2),
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        label_color,
        thickness,
        lineType=cv2.LINE_AA,
    )
    cv2.putText(
        legend,
        f"{u_min:.3f}",
        (2, height - 4),
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        label_color,
        thickness,
        lineType=cv2.LINE_AA,
    )

    return legend



In [ ]:
# --- Updated helpers (override previous definitions) ---

def _collect_track_history(
    meta: Dict[str, List[np.ndarray]],
    track_id: int,
    frame_indices: Sequence[int],
) -> List[Tuple[Optional[np.ndarray], bool, Optional[float]]]:
    history: List[Tuple[Optional[np.ndarray], bool, Optional[float]]] = []
    for idx in frame_indices:
        points = meta["track2d"][idx]
        if track_id >= points.shape[0]:
            history.append((None, False, None))
            continue

        visibles = meta["visibles"][idx]
        if len(visibles) == 0:
            is_visible = True
        else:
            vis_arr = visibles.astype(bool)
            is_visible = track_id < len(vis_arr) and vis_arr[track_id]

        uncertainties = meta["uncertainties"][idx]
        if uncertainties is not None and len(uncertainties) > track_id:
            u_val = uncertainties[track_id]
            if isinstance(u_val, np.ndarray):
                u_val = float(u_val.ravel()[0]) if u_val.size > 0 else None
            elif u_val is not None:
                u_val = float(u_val)
        else:
            u_val = None

        history.append((points[track_id], is_visible, u_val))
    return history


def _history_to_segments(
    history: Sequence[Tuple[Optional[np.ndarray], bool, Optional[float]]]
) -> tuple[list[Tuple[np.ndarray, np.ndarray]], Optional[Tuple[np.ndarray, bool, Optional[float]]]]:
    segments: list[Tuple[np.ndarray, np.ndarray]] = []
    buffer_pts: list[np.ndarray] = []
    buffer_unc: list[float] = []
    last_state: Optional[Tuple[np.ndarray, bool, Optional[float]]] = None

    def _flush():
        nonlocal buffer_pts, buffer_unc
        if len(buffer_pts) >= 2:
            segments.append((np.stack(buffer_pts), np.array(buffer_unc, dtype=float)))
        buffer_pts = []
        buffer_unc = []

    for point, visible, uncertainty in history:
        if point is None:
            _flush()
            continue

        if visible:
            buffer_pts.append(point)
            buffer_unc.append(np.nan if uncertainty is None else float(uncertainty))
        else:
            _flush()

        last_state = (point, visible, uncertainty)

    _flush()

    return segments, last_state


def generate_trace_video(
    meta: Dict[str, List[np.ndarray]],
    image_path: str | Path | None,
    start_idx: int,
    num_frames: int,
    track_ids: Sequence[int],
    color_map: Optional[Dict[int, np.ndarray]] = None,
    output_path: Path = TRACE_VIDEO_OUTPUT,
    fps: int = TRACE_VIDEO_FPS,
    line_width: int = TRACE_VIDEO_LINE_WIDTH,
    last_point_radius: int = TRACE_VIDEO_POINT_SIZE,
    *,
    color_by_uncertainty: bool = False,
    uncertainty_limits: Optional[Tuple[float, float]] = None,
    uncertainty_cmap: str = "viridis",
):
    frame_range = list(range(start_idx, min(start_idx + num_frames, len(meta["track2d"]))))
    if len(frame_range) == 0:
        raise ValueError("No frames available for trace video generation.")

    track_ids = np.asarray(track_ids, dtype=int)
    if track_ids.size == 0:
        track_ids = np.arange(meta["track2d"][start_idx].shape[0])

    if color_map is None:
        color_map = make_color_map(track_ids)

    if color_by_uncertainty:
        cmap = plt.get_cmap(uncertainty_cmap)
        if uncertainty_limits is None:
            limits = _compute_uncertainty_limits(meta, track_ids, frame_range)
        else:
            limits = uncertainty_limits
        legend_width_px = TRACE_UNCERT_LEGEND_WIDTH
        legend_margin_px = TRACE_UNCERT_LEGEND_MARGIN
        legend_font_scale = TRACE_UNCERT_LEGEND_FONT_SCALE
    else:
        cmap = None
        limits = None
        legend_width_px = 0
        legend_margin_px = 0
        legend_font_scale = 0.0

    first_frame = load_rgb_frame(image_path, int(meta["frame_ids"][frame_range[0]]))
    height, width = first_frame.shape[:2]
    total_width = width + (legend_margin_px + legend_width_px if legend_width_px > 0 else 0)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (total_width, height))

    try:
        for upto in range(len(frame_range)):
            frame_idx = frame_range[upto]
            frame_id = int(meta["frame_ids"][frame_idx]) if meta["frame_ids"].size else frame_idx
            image = load_rgb_frame(image_path, frame_id)

            canvas = np.zeros((height, total_width, 3), dtype=np.uint8)
            canvas[:, :width] = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            for tid in track_ids:
                history = _collect_track_history(meta, tid, frame_range[: upto + 1])
                segments, last_state = _history_to_segments(history)
                if len(segments) == 0 and last_state is None:
                    continue

                def _segment_color(u_val):
                    if color_by_uncertainty:
                        return _uncertainty_color(u_val, cmap, limits)
                    return color_map.get(tid, np.array([1.0, 0.2, 0.2]))

                static_color_rgb = color_map.get(tid, np.array([1.0, 0.2, 0.2]))
                static_color_bgr = tuple(int(255 * c) for c in static_color_rgb[::-1])

                for seg_points, seg_unc in segments:
                    if color_by_uncertainty:
                        for i in range(len(seg_points) - 1):
                            u_pair = seg_unc[i : i + 2]
                            valid = [v for v in u_pair if np.isfinite(v)]
                            u_val = valid[0] if valid else None
                            color_rgb = _segment_color(u_val)
                            color_bgr = tuple(int(255 * c) for c in color_rgb[::-1])
                            pt0 = tuple(seg_points[i].round().astype(int))
                            pt1 = tuple(seg_points[i + 1].round().astype(int))
                            cv2.line(
                                canvas,
                                pt0,
                                pt1,
                                color=color_bgr,
                                thickness=line_width,
                                lineType=cv2.LINE_AA,
                            )
                    else:
                        pts = seg_points.reshape(-1, 1, 2).astype(np.int32)
                        cv2.polylines(
                            canvas,
                            [pts],
                            isClosed=False,
                            color=static_color_bgr,
                            thickness=line_width,
                            lineType=cv2.LINE_AA,
                        )

                if last_state is not None:
                    last_point, is_visible, last_unc = last_state
                    if color_by_uncertainty:
                        color_rgb = _segment_color(last_unc)
                        color_bgr = tuple(int(255 * c) for c in color_rgb[::-1])
                    else:
                        color_bgr = static_color_bgr

                    center = tuple(last_point.round().astype(int))
                    if is_visible:
                        cv2.circle(
                            canvas,
                            center,
                            radius=last_point_radius,
                            color=color_bgr,
                            thickness=-1,
                            lineType=cv2.LINE_AA,
                        )
                    else:
                        cv2.circle(
                            canvas,
                            center,
                            radius=max(2, last_point_radius - 2),
                            color=color_bgr,
                            thickness=2,
                            lineType=cv2.LINE_AA,
                        )
                        offset = max(3, last_point_radius - 1)
                        cv2.line(
                            canvas,
                            (center[0] - offset, center[1] - offset),
                            (center[0] + offset, center[1] + offset),
                            color=color_bgr,
                            thickness=2,
                            lineType=cv2.LINE_AA,
                        )
                        cv2.line(
                            canvas,
                            (center[0] - offset, center[1] + offset),
                            (center[0] + offset, center[1] - offset),
                            color=color_bgr,
                            thickness=2,
                            lineType=cv2.LINE_AA,
                        )

            if color_by_uncertainty and legend_width_px > 0:
                legend = _create_uncertainty_legend(
                    height,
                    legend_width_px,
                    cmap,
                    limits,
                    legend_font_scale=legend_font_scale,
                )
                start_x = width + legend_margin_px
                canvas[:, start_x : start_x + legend_width_px] = legend

            writer.write(canvas)
    finally:
        writer.release()

    print(f"Trace video saved to {output_path} ({len(frame_range)} frames @ {fps} fps)")



In [ ]:
def generate_uncertainty_trace_video(
    meta: Dict[str, List[np.ndarray]],
    image_path: str | Path | None,
    start_idx: int,
    num_frames: int,
    track_ids: Sequence[int],
    output_path: Path = TRACE_UNCERT_VIDEO_OUTPUT,
    fps: int = TRACE_VIDEO_FPS,
    line_width: int = TRACE_VIDEO_LINE_WIDTH,
    last_point_radius: int = TRACE_VIDEO_POINT_SIZE,
    legend_width: int = TRACE_UNCERT_LEGEND_WIDTH,
    legend_margin: int = TRACE_UNCERT_LEGEND_MARGIN,
    legend_font_scale: float = TRACE_UNCERT_LEGEND_FONT_SCALE,
    cmap_name: str = TRACE_UNCERT_CMAP,
    color_limits: Optional[Tuple[float, float]] = TRACE_UNCERT_COLOR_LIMITS,
):
    frame_range = list(range(start_idx, min(start_idx + num_frames, len(meta["track2d"]))))
    if len(frame_range) == 0:
        raise ValueError("No frames available for trace video generation.")

    track_ids = np.asarray(track_ids, dtype=int)
    if track_ids.size == 0:
        track_ids = np.arange(meta["track2d"][start_idx].shape[0])

    if color_limits is None:
        limits = _compute_uncertainty_limits(meta, track_ids, frame_range)
    else:
        limits = color_limits

    cmap = plt.get_cmap(cmap_name)

    first_frame = load_rgb_frame(image_path, int(meta["frame_ids"][frame_range[0]]))
    height, width = first_frame.shape[:2]
    total_width = width + legend_margin + legend_width

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (total_width, height))

    def collect_history(tid, upto_idx):
        history = []
        for idx in frame_range[: upto_idx + 1]:
            points = meta["track2d"][idx]
            if tid >= points.shape[0]:
                history.append((None, False, None))
                continue

            visibles = meta["visibles"][idx]
            if len(visibles) == 0:
                is_visible = True
            else:
                vis_arr = visibles.astype(bool)
                is_visible = tid < len(vis_arr) and vis_arr[tid]

            uncertainties = meta["uncertainties"][idx]
            if uncertainties is not None and len(uncertainties) > tid:
                u_val = uncertainties[tid]
                if isinstance(u_val, np.ndarray):
                    u_val = float(u_val.ravel()[0]) if u_val.size > 0 else None
                elif u_val is not None:
                    u_val = float(u_val)
            else:
                u_val = None

            history.append((points[tid], is_visible, u_val))
        return history

    def history_to_segments(history):
        segments = []
        buffer_pts = []
        buffer_unc = []
        last_state = None

        def flush():
            nonlocal buffer_pts, buffer_unc
            if len(buffer_pts) >= 2:
                segments.append((np.stack(buffer_pts), np.array(buffer_unc, dtype=float)))
            buffer_pts = []
            buffer_unc = []

        for point, visible, uncertainty in history:
            if point is None:
                flush()
                continue

            if visible:
                buffer_pts.append(point)
                buffer_unc.append(np.nan if uncertainty is None else float(uncertainty))
            else:
                flush()

            last_state = (point, visible, uncertainty)

        flush()
        return segments, last_state

    try:
        for upto in range(len(frame_range)):
            frame_idx = frame_range[upto]
            frame_id = int(meta["frame_ids"][frame_idx]) if meta["frame_ids"].size else frame_idx
            image = load_rgb_frame(image_path, frame_id)

            canvas = np.zeros((height, total_width, 3), dtype=np.uint8)
            canvas[:, :width] = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            for tid in track_ids:
                history = collect_history(tid, upto)
                segments, last_state = history_to_segments(history)
                if len(segments) == 0 and last_state is None:
                    continue

                for seg_points, seg_unc in segments:
                    for i in range(len(seg_points) - 1):
                        u_pair = seg_unc[i : i + 2]
                        valid = [v for v in u_pair if np.isfinite(v)]
                        u_val = valid[0] if valid else None
                        color_rgb = _uncertainty_color(u_val, cmap, limits)
                        color_bgr = tuple(int(255 * c) for c in color_rgb[::-1])
                        pt0 = tuple(seg_points[i].round().astype(int))
                        pt1 = tuple(seg_points[i + 1].round().astype(int))
                        cv2.line(
                            canvas,
                            pt0,
                            pt1,
                            color=color_bgr,
                            thickness=line_width,
                            lineType=cv2.LINE_AA,
                        )

                if last_state is not None:
                    last_point, is_visible, last_unc = last_state
                    color_rgb = _uncertainty_color(last_unc, cmap, limits)
                    color_bgr = tuple(int(255 * c) for c in color_rgb[::-1])
                    center = tuple(last_point.round().astype(int))
                    if is_visible:
                        cv2.circle(
                            canvas,
                            center,
                            radius=last_point_radius,
                            color=color_bgr,
                            thickness=-1,
                            lineType=cv2.LINE_AA,
                        )
                    else:
                        cv2.circle(
                            canvas,
                            center,
                            radius=max(2, last_point_radius - 2),
                            color=color_bgr,
                            thickness=2,
                            lineType=cv2.LINE_AA,
                        )
                        offset = max(3, last_point_radius - 1)
                        cv2.line(
                            canvas,
                            (center[0] - offset, center[1] - offset),
                            (center[0] + offset, center[1] + offset),
                            color=color_bgr,
                            thickness=2,
                            lineType=cv2.LINE_AA,
                        )
                        cv2.line(
                            canvas,
                            (center[0] - offset, center[1] + offset),
                            (center[0] + offset, center[1] - offset),
                            color=color_bgr,
                            thickness=2,
                            lineType=cv2.LINE_AA,
                        )

            legend = _create_uncertainty_legend(
                height,
                legend_width,
                cmap,
                limits,
                font_scale=legend_font_scale,
            )
            start_x = width + legend_margin
            canvas[:, start_x : start_x + legend_width] = legend

            writer.write(canvas)
    finally:
        writer.release()

    print(f"Uncertainty trace video saved to {output_path} ({len(frame_range)} frames @ {fps} fps)")



In [ ]:
def _compute_uncertainty_limits(
    meta: Dict[str, List[np.ndarray]],
    track_ids: Sequence[int],
    frame_indices: Sequence[int],
) -> Tuple[float, float]:
    values = []
    for idx in frame_indices:
        arr = meta["uncertainties"][idx]
        if arr is None or len(arr) == 0:
            continue
        for tid in track_ids:
            if tid < len(arr):
                val = arr[tid]
                if val is not None and np.isfinite(val):
                    values.append(float(val))
    if not values:
        return 0.0, 1.0
    return float(np.min(values)), float(np.max(values))


def _uncertainty_color(value: Optional[float], cmap, limits: Tuple[float, float]) -> Tuple[float, float, float]:
    if value is None or not np.isfinite(value):
        return (0.8, 0.8, 0.8)
    u_min, u_max = limits
    if u_max - u_min < 1e-8:
        norm = 0.5
    else:
        norm = (value - u_min) / (u_max - u_min)
    norm = float(np.clip(norm, 0.0, 1.0))
    color = cmap(norm)[:3]
    return color



In [ ]:
def _collect_track_history(
    meta: Dict[str, List[np.ndarray]],
    track_id: int,
    frame_indices: Sequence[int],
) -> List[Tuple[Optional[np.ndarray], bool, Optional[float]]]:
    history: List[Tuple[Optional[np.ndarray], bool, Optional[float]]] = []
    for idx in frame_indices:
        points = meta["track2d"][idx]
        if track_id >= points.shape[0]:
            history.append((None, False, None))
            continue

        visibles = meta["visibles"][idx]
        if len(visibles) == 0:
            is_visible = True
        else:
            vis_arr = visibles.astype(bool)
            is_visible = track_id < len(vis_arr) and vis_arr[track_id]

        uncertainties = meta["uncertainties"][idx]
        if uncertainties is not None and len(uncertainties) > track_id:
            u_val = uncertainties[track_id]
            if isinstance(u_val, np.ndarray):
                if u_val.size > 0:
                    u_val = float(np.ravel(u_val)[0])
                else:
                    u_val = None
            elif u_val is not None:
                u_val = float(u_val)
        else:
            u_val = None

        history.append((points[track_id], is_visible, u_val))
    return history


def _history_to_segments(
    history: Sequence[Tuple[Optional[np.ndarray], bool, Optional[float]]]
) -> tuple[list[Tuple[np.ndarray, np.ndarray]], Optional[Tuple[np.ndarray, bool, Optional[float]]]]:
    segments: list[Tuple[np.ndarray, np.ndarray]] = []
    buffer_pts: list[np.ndarray] = []
    buffer_unc: list[float] = []
    last_state: Optional[Tuple[np.ndarray, bool, Optional[float]]] = None

    def _flush():
        nonlocal buffer_pts, buffer_unc
        if len(buffer_pts) >= 2:
            segments.append((np.stack(buffer_pts), np.array(buffer_unc, dtype=float)))
        buffer_pts = []
        buffer_unc = []

    for point, visible, uncertainty in history:
        if point is None:
            _flush()
            continue

        if visible:
            buffer_pts.append(point)
            buffer_unc.append(np.nan if uncertainty is None else float(uncertainty))
        else:
            _flush()

        last_state = (point, visible, uncertainty)

    _flush()

    return segments, last_state


def generate_trace_video(
    meta: Dict[str, List[np.ndarray]],
    image_path: str | Path | None,
    start_idx: int,
    num_frames: int,
    track_ids: Sequence[int],
    color_map: Optional[Dict[int, np.ndarray]] = None,
    output_path: Path = TRACE_VIDEO_OUTPUT,
    fps: int = TRACE_VIDEO_FPS,
    line_width: int = TRACE_VIDEO_LINE_WIDTH,
    last_point_radius: int = TRACE_VIDEO_POINT_SIZE,
    *,
    color_by_uncertainty: bool = False,
    uncertainty_limits: Optional[Tuple[float, float]] = None,
    uncertainty_cmap: str = "viridis",
):
    frame_range = list(range(start_idx, min(start_idx + num_frames, len(meta["track2d"]))))
    if len(frame_range) == 0:
        raise ValueError("No frames available for trace video generation.")

    track_ids = np.asarray(track_ids, dtype=int)
    if track_ids.size == 0:
        track_ids = np.arange(meta["track2d"][start_idx].shape[0])

    if color_map is None:
        color_map = make_color_map(track_ids)

    if color_by_uncertainty:
        cmap = plt.get_cmap(uncertainty_cmap)
        if uncertainty_limits is None:
            limits = _compute_uncertainty_limits(meta, track_ids, frame_range)
        else:
            limits = uncertainty_limits
    else:
        cmap = None
        limits = None

    first_frame = load_rgb_frame(image_path, int(meta["frame_ids"][frame_range[0]]))
    height, width = first_frame.shape[:2]

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    try:
        for upto in range(len(frame_range)):
            frame_idx = frame_range[upto]
            frame_id = int(meta["frame_ids"][frame_idx]) if meta["frame_ids"].size else frame_idx
            image = load_rgb_frame(image_path, frame_id)
            canvas = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            for tid in track_ids:
                history = _collect_track_history(meta, tid, frame_range[: upto + 1])
                segments, last_state = _history_to_segments(history)
                if len(segments) == 0 and last_state is None:
                    continue

                def _segment_color(u_val):
                    if color_by_uncertainty:
                        return _uncertainty_color(u_val, cmap, limits)
                    return color_map.get(tid, np.array([1.0, 0.2, 0.2]))

                if not color_by_uncertainty:
                    color_rgb = color_map.get(tid, np.array([1.0, 0.2, 0.2]))
                    color_bgr_static = tuple(int(255 * c) for c in color_rgb[::-1])
                else:
                    color_bgr_static = None

                for seg_points, seg_unc in segments:
                    if color_by_uncertainty:
                        for i in range(len(seg_points) - 1):
                            u_pair = seg_unc[i : i + 2]
                            valid = [v for v in u_pair if np.isfinite(v)]
                            u_val = valid[0] if valid else None
                            color_rgb = _segment_color(u_val)
                            color_bgr = tuple(int(255 * c) for c in color_rgb[::-1])
                            pt0 = tuple(seg_points[i].round().astype(int))
                            pt1 = tuple(seg_points[i + 1].round().astype(int))
                            cv2.line(
                                canvas,
                                pt0,
                                pt1,
                                color=color_bgr,
                                thickness=line_width,
                                lineType=cv2.LINE_AA,
                            )
                    else:
                        pts = seg_points.reshape(-1, 1, 2).astype(np.int32)
                        cv2.polylines(
                            canvas,
                            [pts],
                            isClosed=False,
                            color=color_bgr_static,
                            thickness=line_width,
                            lineType=cv2.LINE_AA,
                        )

                if last_state is not None:
                    last_point, is_visible, last_unc = last_state
                    if color_by_uncertainty:
                        color_rgb = _segment_color(last_unc)
                        color_bgr = tuple(int(255 * c) for c in color_rgb[::-1])
                    else:
                        color_rgb = color_map.get(tid, np.array([1.0, 0.2, 0.2]))
                        color_bgr = tuple(int(255 * c) for c in color_rgb[::-1])

                    center = tuple(last_point.round().astype(int))
                    if is_visible:
                        cv2.circle(
                            canvas,
                            center,
                            radius=last_point_radius,
                            color=color_bgr,
                            thickness=-1,
                            lineType=cv2.LINE_AA,
                        )
                    else:
                        cv2.circle(
                            canvas,
                            center,
                            radius=max(2, last_point_radius - 2),
                            color=color_bgr,
                            thickness=2,
                            lineType=cv2.LINE_AA,
                        )
                        offset = max(3, last_point_radius - 1)
                        cv2.line(
                            canvas,
                            (center[0] - offset, center[1] - offset),
                            (center[0] + offset, center[1] + offset),
                            color=color_bgr,
                            thickness=2,
                            lineType=cv2.LINE_AA,
                        )
                        cv2.line(
                            canvas,
                            (center[0] - offset, center[1] + offset),
                            (center[0] + offset, center[1] - offset),
                            color=color_bgr,
                            thickness=2,
                            lineType=cv2.LINE_AA,
                        )

            writer.write(canvas)
    finally:
        writer.release()

    print(f"Trace video saved to {output_path} ({len(frame_range)} frames @ {fps} fps)")



In [ ]:
meta = load_meta_data(META_DATA_PATH)
frame_ids = meta["frame_ids"]
num_frames_available = len(meta["track2d"])
unique_points_start = meta["track2d"][0].shape[0] if num_frames_available else 0

print(f"Loaded {num_frames_available} frames from meta_data")
if frame_ids.size:
    print(f"Frame id range: {frame_ids.min()} → {frame_ids.max()}")
print(f"Points in frame 0: {unique_points_start}")


In [ ]:
start_index = get_start_index(frame_ids, START_FRAME, TREAT_START_AS_FRAME_ID)
selected_track_ids = choose_track_ids(meta, start_index, FILTER_POINTS_BY_START_VISIBILITY)
color_map = make_color_map(selected_track_ids)

print(f"Start index: {start_index} (frame id {int(frame_ids[start_index]) if frame_ids.size else start_index})")
print(f"Tracking {len(selected_track_ids)} points")
if len(selected_track_ids) > 0:
    preview = ", ".join(map(str, selected_track_ids[:10]))
    print(f"Sample track ids: {preview}{'…' if len(selected_track_ids) > 10 else ''}")


In [ ]:
visualize_sequence(
    meta,
    IMAGE_PATH_TEMPLATE,
    start_index,
    NUM_FRAMES,
    selected_track_ids,
    point_size=POINT_SIZE,
    alpha=POINT_ALPHA,
    edge_color=POINT_EDGE_COLOR,
    max_cols=MAX_COLUMNS,
)


In [ ]:
generate_uncertainty_trace_video(
    meta,
    IMAGE_PATH_TEMPLATE,
    start_index,
    NUM_FRAMES,
    selected_track_ids,
    output_path=TRACE_UNCERT_VIDEO_OUTPUT,
    fps=TRACE_VIDEO_FPS,
    line_width=TRACE_VIDEO_LINE_WIDTH,
    last_point_radius=TRACE_VIDEO_POINT_SIZE,
    legend_width=TRACE_UNCERT_LEGEND_WIDTH,
    legend_margin=TRACE_UNCERT_LEGEND_MARGIN,
    legend_font_scale=TRACE_UNCERT_LEGEND_FONT_SCALE,
    cmap_name=TRACE_UNCERT_CMAP,
    color_limits=TRACE_UNCERT_COLOR_LIMITS,
)



In [ ]:
generate_trace_video(
    meta,
    IMAGE_PATH_TEMPLATE,
    start_index,
    NUM_FRAMES,
    selected_track_ids,
    color_map=color_map,
    output_path=TRACE_VIDEO_OUTPUT,
    fps=TRACE_VIDEO_FPS,
    line_width=TRACE_VIDEO_LINE_WIDTH,
    last_point_radius=TRACE_VIDEO_POINT_SIZE,
)



In [ ]:
plot_traces_on_last_frame(
    meta,
    IMAGE_PATH_TEMPLATE,
    start_index,
    NUM_FRAMES,
    selected_track_ids,
    color_map=color_map,
    linewidth=2.5,
    point_size=POINT_SIZE * 1.2,
    alpha=0.9,
)

